# Big Data HS 2020 - Cheat Sheet

During the exam, you will be required to write SQL, Spark RDD, Spark SQL and JSONiq queries using a Jupyter notebook.

This notebook is designed to help you start writing your queries by providing you an environment with the datasets loaded and a simple query that you can use to recap the syntax of each query language.

Feel free to extend this notebook and use it for preparing the answers you need for the exam. Take into attention that the content of this notebook will not be considered for grading.

## SQL

There is a local PostgreSQL 12 installation with a dataset loaded into a database. Run the next cell to connect to it.

In [ ]:
%load_ext sql
%sql postgresql://postgres:example@db

To print the tables currently loaded in the database run:

In [ ]:
%%sql

SELECT * 
FROM INFORMATION_SCHEMA.TABLES 
WHERE TABLE_TYPE = 'BASE TABLE' and TABLE_CATALOG = 'postgres' and TABLE_SCHEMA = 'public';

To print the attributes of a particular table ('companies', for example) run:

In [ ]:
%%sql

SELECT column_name, data_type, character_maximum_length
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE table_name = 'departments';

A query against the given database could look like this:

In [ ]:
%%sql

SELECT
    dept_no, dept_name, MIN(birth_date)
FROM employees AS e
JOIN
    dept_emp AS de USING (emp_no)
JOIN
    departments AS d USING (dept_no)
WHERE
    to_date = '9999-01-01'
GROUP BY dept_no, dept_name
ORDER BY MIN(birth_date)
LIMIT 3

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now it's your turn: you can write all your queries in new cells below. Feel free to add as many cells as needed.

## Spark

For these questions you are required to write Spark queries on a provided student dataset.

 To set up Spark, run the following cell:

In [ ]:
import json

from pyspark import SparkContext
from jsoniq import RumbleSession

spark = RumbleSession \
    .builder \
    .appName("Big Data examination") \
    .getOrCreate()

sc = spark.sparkContext

### Data set

We will use a data set that contains an array of students with respective grades, each student has all of the following attributes:

- `first`: string containing the first name of the student.
- `last`: string containing the last name of the student.
- `university`: string representing the university. Possible values are "ETH" and "EPFL".
- `grades`: a dictionary whose key-value pairs are combinations of subjects with the respective grade, each student has from 1 to 6 grades, all in different subjects, the possible subjects are: Math, Biology, Chemistry, Philosophy, History, Literature. Each grade is an integer between 0 and 100.

You can use the RDD interface or the DataFrame interface below.

### RDD interface

Run the code below to import the JSON lines file as an RDD:

In [ ]:
students = sc.textFile("students.jsonl").map(json.loads)

Inspect the data set by printing the first row:

In [ ]:
students.take(1)

Below are some examples of Spark queries on the students data set.

In [ ]:
students.map(lambda s: s["first"]).take(3)

In [ ]:
students.filter(lambda s: s["first"][0] == "K").take(1)

In [ ]:
students.flatMap(lambda s: s["grades"]).take(3)

In [ ]:
students.count()

### DataFrame interface

Run the code below to create a Spark DataFrame:

In [ ]:
%load_ext sparksql_magic

students_df = spark.read.json('students.jsonl')
students_df.createOrReplaceTempView("students")

Print the type and schema:


In [ ]:
print(type(students_df))
students_df.printSchema()

Print one row using spark dataframe.

In [ ]:
students_df.limit(1).collect()

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now it's your turn: you can write all your queries in new cells below. Feel free to add as many cells as needed.

## JSONiq (Rumble)

Run the two cells below to connect to your Rumble server.

In [ ]:
%load_ext jsoniqmagic

Github Archive is a project to record the public GitHub timeline. For the following exam questions, we will be working the `git-archive.json` data set that provides a subset of all records of the public GitHub timeline.

## General Queries

To print the data set, use *json-file()*.

In [ ]:
%%jsoniq
json-file("git-archive.json", 10)[1]

To see the fields, use the *keys()* function.

In [ ]:
%%jsoniq
keys(json-file("git-archive.json", 10))

You can display the first objects with the *count* clause.

In [ ]:
%%jsoniq -pdf
for $event in json-file("git-archive.json", 10)
count $c
where $c le 10
return $event

You can use . and [] for navigation (in parallel).

In [ ]:
%%jsoniq
distinct-values(json-file("git-archive.json", 10).payload.commits[].author.email)

In [ ]:
%%jsoniq
json-file("git-archive.json", 10).payload.commits[[1]].author.name

A more complex query with FLWOR:

In [ ]:
%%jsoniq -pdf

for $event in json-file("git-archive.json", 10)
let $actor := $event.actor.id
group by $actor
let $nb_commits := count($event.payload.commits[])
order by $nb_commits descending
count $n
where $n le 15
return {
    "actor" : $actor,
    "number of commits" : $nb_commits,
    "first commit email" : $event[1].payload.commits[[1]].author.email
}

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now it's your turn: you can write all your queries in new cells below. Feel free to add as many cells as needed.